In [13]:
import pandas as pd

In [14]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Forme :", df.shape)
df.head()

Forme : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [15]:
def audit_qualite(df):
    print("Forme :", df.shape)
    print("\nTypes :")
    print(df.dtypes)
    print("\nPourcentage de valeurs manquantes :")
    print((df.isna().mean() * 100).sort_values(ascending=False))
    print("\nChurn en nombre :")
    print(df["Churn"].value_counts())
    print("\nChurn en pourcentage :")
    print(df["Churn"].value_counts(normalize=True) * 100)

In [16]:
audit_qualite(df)

Forme : (7043, 21)

Types :
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Pourcentage de valeurs manquantes :
customerID          0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
OnlineBackup        0.0
gender              0.0
OnlineSecurity      0.0
InternetService     0

In [17]:
audit_qualite(df[df["Churn"] == "No"])

Forme : (5174, 21)

Types :
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Pourcentage de valeurs manquantes :
customerID          0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
OnlineBackup        0.0
gender              0.0
OnlineSecurity      0.0
InternetService     0

# Phase 2 

In [18]:
def reparer_total_charges(df):
    df_repare = df.copy()
    total_numerique = pd.to_numeric(df_repare["TotalCharges"], errors="coerce")

    if total_numerique.notna().sum() == 0:
        raise ValueError("TotalCharges ne contient aucune valeur numérique.")

    valeurs_invalides = df_repare.loc[total_numerique.isna(), "TotalCharges"].unique()
    print("Valeurs illisibles trouvées :", valeurs_invalides)
    print("Trous révélés :", total_numerique.isna().sum())

    df_repare["TotalCharges"] = total_numerique.fillna(total_numerique.median())
    return df_repare

In [19]:
df_repare = reparer_total_charges(df)
print("Type après réparation :", df_repare["TotalCharges"].dtype)
print("Trous restants :", df_repare["TotalCharges"].isna().sum())

Valeurs illisibles trouvées : <StringArray>
[' ']
Length: 1, dtype: str
Trous révélés : 11
Type après réparation : float64
Trous restants : 0


In [20]:
df_texte = df.copy()
df_texte["TotalCharges"] = "texte"

try:
    reparer_total_charges(df_texte)
except ValueError as erreur:
    print("Erreur détectée :", erreur)

Erreur détectée : TotalCharges ne contient aucune valeur numérique.


In [21]:
df_virgule = df.head(3).copy()
df_virgule.loc[df_virgule.index[0], "TotalCharges"] = "29,90"
reparer_total_charges(df_virgule)

Valeurs illisibles trouvées : <StringArray>
['29,90']
Length: 1, dtype: str
Trous révélés : 1


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,998.825,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.500,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.150,Yes
